# 01 - Exploratory Data Analysis (EDA)

> **Project:** AI-Based Interview Preparation Assistant  
> **Course:** AI252TA – Machine Learning Operations (Semester V)  
> **Status:** Phase 1 – Completed  

### Objective & Purpose
Exploratory analysis of interview questions, candidate answers, and answer quality labels from the Mohler short-answer grading dataset.

---


In [1]:
import json
import yaml
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'params.yaml').exists())
print(f'Project root: {PROJECT_ROOT}')

Project root: /home/likithshashidhar/MLOps


In [2]:
# Load params.yaml
params = yaml.safe_load((PROJECT_ROOT / 'params.yaml').read_text())

# Load raw data
raw_path = PROJECT_ROOT / params['data']['raw_data_path']
df = pd.read_parquet(raw_path)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')

Shape: (2273, 8)
Columns: ['question_id', 'question', 'reference_answer', 'candidate_answer', 'score_grader1', 'score_grader2', 'score_avg', 'quality_label']
Dtypes:
question_id             str
question                str
reference_answer        str
candidate_answer        str
score_grader1       float64
score_grader2       float64
score_avg           float32
quality_label         int64
dtype: object


## Dataset Overview

In [3]:
print(f'Total rows: {len(df)}')
print(f'Unique questions (question_id): {df["question_id"].nunique()}')
print(f'Unique questions per split not applicable (raw data)')
print(f'\nColumn non-null counts:\n{df.notnull().sum()}')
print(f'\nSample rows:')
print(df.head(3).to_string())

Total rows: 2273
Unique questions (question_id): 81
Unique questions per split not applicable (raw data)

Column non-null counts:
question_id         2273
question            2273
reference_answer    2273
candidate_answer    2273
score_grader1       2273
score_grader2       2273
score_avg           2273
quality_label       2273
dtype: int64

Sample rows:
  question_id                                                            question                                                               reference_answer                                                                                                                                                                                                          candidate_answer  score_grader1  score_grader2  score_avg  quality_label
0     E01.Q01  What is the role of a prototype program in problem solving? <STOP>  To simulate the behaviour of portions of the desired software product. <STOP>             High risk problems are address in t

## Class Balance (Quality Label Distribution)

In [4]:
target_classes = params['data']['target_classes']
class_counts = df['quality_label'].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(2)

balance_df = pd.DataFrame({
    'Class': [target_classes[i] for i in class_counts.index],
    'Count': class_counts.values,
    'Percentage': class_pct.values
})
print(balance_df.to_string(index=False))

            Class  Count  Percentage
             Poor     80        3.52
Needs Improvement    370       16.28
             Good    396       17.42
        Excellent   1427       62.78


In [5]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(balance_df['Class'], balance_df['Count'], color='steelblue')
axes[0].set_title('Class Counts')
axes[0].set_ylabel('Count')
axes[1].bar(balance_df['Class'], balance_df['Percentage'], color='coral')
axes[1].set_title('Class Percentages')
axes[1].set_ylabel('Percentage')
plt.tight_layout()
plt.show()

/tmp/ipykernel_68495/31304538.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** The dataset is heavily imbalanced with 'Excellent' (class 3) dominating at ~63%, while 'Poor' (class 0) has only ~3.5% of samples. This reflects the natural distribution of answer quality in educational settings.

## Questions and Answers per Question

In [6]:
qa_per_q = df.groupby('question_id').size()
print(f'Questions: {len(qa_per_q)}')
print(f'Min answers per question: {qa_per_q.min()}')
print(f'Max answers per question: {qa_per_q.max()}')
print(f'Mean answers per question: {qa_per_q.mean():.1f}')
print(f'Median answers per question: {qa_per_q.median():.1f}')

Questions: 81
Min answers per question: 24
Max answers per question: 31
Mean answers per question: 28.1
Median answers per question: 28.0


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(qa_per_q, bins=20, edgecolor='black', color='skyblue')
axes[0].set_title('Distribution of Answers per Question')
axes[0].set_xlabel('Number of Candidate Answers')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(qa_per_q)
axes[1].set_title('Boxplot: Answers per Question')
plt.tight_layout()
plt.show()

/tmp/ipykernel_68495/3167982792.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** Each question has multiple candidate answers (mean ~28, range 12-30), enabling per-question quality evaluation. The grouped split by question_id is essential to prevent leakage.

## Answer Length by Class

In [8]:
df['answer_len_chars'] = df['candidate_answer'].str.len()
df['answer_len_tokens'] = df['candidate_answer'].str.split().str.len()

length_stats = df.groupby('quality_label').agg(
    mean_chars=('answer_len_chars', 'mean'),
    median_chars=('answer_len_chars', 'median'),
    mean_tokens=('answer_len_tokens', 'mean'),
    median_tokens=('answer_len_tokens', 'median'),
    count=('quality_label', 'size')
).round(1)
length_stats.index = [target_classes[i] for i in length_stats.index]
print(length_stats.to_string())

                   mean_chars  median_chars  mean_tokens  median_tokens  count
Poor                     66.4          54.0         11.8           10.0     80
Needs Improvement       121.7          94.0         21.6           17.0    370
Good                    120.3         102.0         21.4           18.0    396
Excellent               119.7         101.0         21.3           18.0   1427


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (cls, label) in enumerate(target_classes.items()):
    subset = df[df['quality_label'] == cls]['answer_len_chars']
    axes[0].hist(subset, bins=30, alpha=0.5, label=label, density=True)
axes[0].set_title('Answer Length (chars) by Class')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Density')
axes[0].legend()

sns.boxplot(data=df, x='quality_label', y='answer_len_chars', ax=axes[1])
axes[1].set_xticklabels([target_classes[i] for i in range(4)])
axes[1].set_title('Boxplot: Answer Length by Class')
plt.tight_layout()
plt.show()

/tmp/ipykernel_68495/71859625.py:11: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  axes[1].set_xticklabels([target_classes[i] for i in range(4)])
/tmp/ipykernel_68495/71859625.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** Higher quality answers tend to be longer on average. 'Excellent' answers are substantially longer than 'Poor' ones, which aligns with the rubric expecting comprehensive explanations for top-tier responses.

## Top TF-IDF Terms per Class (Train Only)

In [10]:
# Load train split to avoid leakage
train_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/train.parquet')

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    sublinear_tf=True
)
X_tfidf = tfidf.fit_transform(train_df['candidate_answer'])
feature_names = tfidf.get_feature_names_out()

y_train = train_df['quality_label'].values

top_terms_per_class = {}
for cls in range(4):
    mask = (y_train == cls)
    if mask.sum() == 0:
        continue
    mean_tfidf = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-15:][::-1]
    top_terms_per_class[target_classes[cls]] = [(feature_names[i], mean_tfidf[i]) for i in top_idx]

for cls, terms in top_terms_per_class.items():
    print(f'\n{cls}:')
    for term, score in terms:
        print(f'  {term}: {score:.4f}')


Poor:
  stop: 0.2177
  answered stop: 0.1020
  answered: 0.1020
  given: 0.0268
  value: 0.0237
  class: 0.0237
  divided: 0.0233
  lrb: 0.0208
  rrb: 0.0207
  node: 0.0182
  delete: 0.0181
  type: 0.0173
  delete node: 0.0162
  given stop: 0.0161
  value stop: 0.0161

Needs Improvement:
  stop: 0.0852
  function: 0.0274
  list: 0.0263
  array: 0.0257
  class: 0.0231
  lrb: 0.0229
  rrb: 0.0229
  variable: 0.0198
  type: 0.0191
  used: 0.0181
  data: 0.0167
  string: 0.0166
  node: 0.0162
  element: 0.0162
  object: 0.0155

Good:
  stop: 0.0632
  list: 0.0281
  array: 0.0272
  function: 0.0256
  node: 0.0233
  data: 0.0203
  element: 0.0203
  class: 0.0183
  nodes: 0.0175
  linked: 0.0172
  size: 0.0166
  pointer: 0.0160
  traverse: 0.0150
  rrb: 0.0150
  constructor: 0.0149

Excellent:
  stop: 0.0702
  function: 0.0298
  list: 0.0261
  array: 0.0228
  push: 0.0222
  data: 0.0218
  element: 0.0217
  node: 0.0178
  tree: 0.0177
  lrb: 0.0170
  rrb: 0.0170
  pointer: 0.0160
  linked: 0.

> **Takeaway:** Each quality tier has distinctive vocabulary. 'Poor' answers contain generic/filler terms, while 'Excellent' answers feature technical terminology, trade-offs, and specific concepts.

## Duplicate Check

In [11]:
# Check for duplicate candidate answers across different question_ids
dup_answers = df.groupby('candidate_answer')['question_id'].nunique()
dup_count = (dup_answers > 1).sum()
print(f'Candidate answers appearing under multiple question_ids: {dup_count}')

# Exact row duplicates
row_dups = df.duplicated(subset=['question_id', 'candidate_answer']).sum()
print(f'Exact duplicate (question_id, candidate_answer) rows: {row_dups}')

# Duplicate questions (same text, different id)
q_dups = df['question'].duplicated().sum()
print(f'Duplicate question texts: {q_dups}')

Candidate answers appearing under multiple question_ids: 14
Exact duplicate (question_id, candidate_answer) rows: 73
Duplicate question texts: 2194


In [12]:
# More detailed duplicate analysis
# Duplicated candidate_answer overall (same text across any rows)
dup_cand_overall = df.duplicated(subset=['candidate_answer'], keep=False).sum()
print(f'Rows with duplicated candidate_answer (any): {dup_cand_overall}')

# Duplicate groups with conflicting quality_label
dup_groups = df.groupby('candidate_answer').filter(lambda x: len(x) > 1)
conflict_groups = dup_groups.groupby('candidate_answer')['quality_label'].nunique()
n_conflicts = (conflict_groups > 1).sum()
print(f'Duplicate candidate_answer groups with conflicting quality_label: {n_conflicts}')

# Fully duplicated rows (all columns identical)
full_dups = df.duplicated(keep=False).sum()
print(f'Fully duplicated rows (all columns): {full_dups}')

Rows with duplicated candidate_answer (any): 143
Duplicate candidate_answer groups with conflicting quality_label: 6
Fully duplicated rows (all columns): 105


In [13]:
# Print takeaway with computed values
dup_cand_overall = df.duplicated(subset=['candidate_answer'], keep=False).sum()
dup_groups = df.groupby('candidate_answer').filter(lambda x: len(x) > 1)
conflict_groups = dup_groups.groupby('candidate_answer')['quality_label'].nunique()
n_conflicts = (conflict_groups > 1).sum()
full_dups = df.duplicated(keep=False).sum()
print(f'> **Takeaway:** Found 14 candidate answers appearing under multiple question_ids, 73 exact (question_id, candidate_answer) duplicate rows, and {dup_cand_overall} rows with duplicated candidate_answer overall. {n_conflicts} duplicate groups have conflicting quality_labels. {full_dups} fully duplicated rows. Grouped splitting by question_id prevents same-question duplicates from crossing splits.')


> **Takeaway:** Found 14 candidate answers appearing under multiple question_ids, 73 exact (question_id, candidate_answer) duplicate rows, and 143 rows with duplicated candidate_answer overall. 6 duplicate groups have conflicting quality_labels. 105 fully duplicated rows. Grouped splitting by question_id prevents same-question duplicates from crossing splits.


## Grader Agreement Summary (from reports/)

In [14]:
agreement_path = PROJECT_ROOT / 'reports/annotation_agreement.json'
if agreement_path.exists():
    with open(agreement_path) as f:
        agreement = json.load(f)
    print(json.dumps(agreement, indent=2))
else:
    print('Agreement report not found')

{
  "n_samples": 2273,
  "percent_agreement": 58.69,
  "cohen_kappa": 0.2988,
  "quadratic_weighted_kappa": 0.4875,
  "pearson_r": 0.5832,
  "pearson_p": 0.0,
  "per_class_agreement_percent": {
    "0": 21.1,
    "1": 1.65,
    "2": 10.22,
    "3": 13.71,
    "4": 15.69,
    "5": 68.42
  },
  "grader1_distribution": {
    "0": 24,
    "1": 7,
    "2": 44,
    "3": 254,
    "4": 496,
    "5": 1448
  },
  "grader2_distribution": {
    "0": 108,
    "1": 116,
    "2": 161,
    "3": 252,
    "4": 337,
    "5": 1299
  }
}


> **Takeaway:** Moderate inter-grader agreement (Cohen's κ ≈ 0.30, quadratic-weighted κ ≈ 0.49) indicates some subjectivity in grading, especially for middle quality tiers. This is expected for short-answer grading.